# Download & Pemotongan Dataset Audio Kaggle (Human vs AI-Generated Voices)
**Sumber Dataset:** [Speech Dataset of Human and AI-Generated Voices (Kaggle)](https://www.kaggle.com/datasets/kambingbersayaphitam/speech-dataset-of-human-and-ai-generated-voices)

**Spesifikasi Target:**
- **Kelas 1 (`asli_human` / `Real`):** 100 sampel $\times$ 30 detik = 3.000 detik (**50 menit**)
- **Kelas 2 (`ai_manipulasi` / `Fake`):** 100 sampel $\times$ 30 detik = 3.000 detik (**50 menit**)
- **Sampling Rate (`sr`):** `22.050 Hz` (atau `16.000 Hz` seragam, Mono)
- **Metode:** Mengunduh file `.wav` berdurasi panjang (4–9 menit per file) langsung dari arsip Kaggle dan memotongnya tepat per **30 detik** menjadi `audio_001.wav` s/d `audio_100.wav`.

In [7]:
import os
import io
import zipfile
import requests
import librosa
import numpy as np
import soundfile as sf

# ==========================================
# 1. KONFIGURASI PATH & TARGET SAMPEL
# ==========================================
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/kambingbersayaphitam/speech-dataset-of-human-and-ai-generated-voices"
FOLDER_ASLI = "../data/dataset_audio_kaggle/asli_human"
FOLDER_AI   = "../data/dataset_audio_kaggle/ai_manipulasi"

os.makedirs(FOLDER_ASLI, exist_ok=True)
os.makedirs(FOLDER_AI, exist_ok=True)

SR_TARGET = 16000                 # Standar 16.000 Hz untuk suara bicara (Speech)
DURASI_PER_FILE = 30              # 30 detik per sampel
SAMPEL_PER_FILE = SR_TARGET * DURASI_PER_FILE  # 661.500 titik sampel per 30 detik
TARGET_JUMLAH_FILE = 100          # 100 file per kelas (total 50 menit per kelas)

# ==========================================
# 2. HELPER STREAMING ZIP DARI KAGGLE
# ==========================================
class HttpZipFile(io.RawIOBase):
    """Membaca file ZIP di Kaggle secara selektif menggunakan HTTP Range Request."""
    def __init__(self, url):
        self.session = requests.Session()
        r = self.session.get(url, stream=True, allow_redirects=True, timeout=60)
        r.raise_for_status()
        self.url = r.url
        self.size = int(r.headers["Content-Length"])
        r.close()
        self.pos = 0

    def seekable(self):
        return True

    def readable(self):
        return True

    def seek(self, offset, whence=0):
        if whence == 0:
            self.pos = offset
        elif whence == 1:
            self.pos += offset
        elif whence == 2:
            self.pos = self.size + offset
        return self.pos

    def tell(self):
        return self.pos

    def read(self, size=-1):
        if size == 0 or self.pos >= self.size:
            return b""
        end = self.size - 1 if size < 0 else min(self.size - 1, self.pos + size - 1)
        for attempt in range(3):
            try:
                r = self.session.get(self.url, headers={"Range": f"bytes={self.pos}-{end}"}, timeout=120)
                r.raise_for_status()
                data = r.content
                self.pos += len(data)
                return data
            except Exception as e:
                if attempt == 2:
                    raise e

def extract_single_member_bytes(http_zip, zip_info):
    """Mengunduh 1 file .wav dari dalam ZIP hanya dengan 1 kali HTTP Request."""
    # Baca local file header (30 bytes) untuk mengetahui panjang nama file & extra field
    http_zip.seek(zip_info.header_offset)
    local_header = http_zip.read(30)
    fname_len = int.from_bytes(local_header[26:28], "little")
    extra_len = int.from_bytes(local_header[28:30], "little")
    data_start = zip_info.header_offset + 30 + fname_len + extra_len
    total_chunk_size = (data_start - zip_info.header_offset) + zip_info.compress_size

    # Unduh seluruh blok file ini sekaligus ke memori lokal
    http_zip.seek(zip_info.header_offset)
    raw_chunk = http_zip.read(total_chunk_size)

    # Decompress menggunakan zlib/zipfile dari memori
    import zlib
    comp_data = raw_chunk[30 + fname_len + extra_len :]
    if zip_info.compress_type == zipfile.ZIP_STORED:
        return comp_data
    elif zip_info.compress_type == zipfile.ZIP_DEFLATED:
        return zlib.decompress(comp_data, -15)
    else:
        raise ValueError(f"Unsupported compression type: {zip_info.compress_type}")

# ==========================================
# 3. PROSES DOWNLOAD & PEMOTONGAN 30 DETIK
# ==========================================
print("Menghubungkan ke server Kaggle...")
raw_http = HttpZipFile(KAGGLE_URL)
zf = zipfile.ZipFile(raw_http)

# Simpan juga Dataset.csv sebagai referensi metadata
csv_info = zf.getinfo("Dataset.csv")
csv_bytes = extract_single_member_bytes(raw_http, csv_info)
with open("../data/dataset_audio_kaggle/Dataset_metadata.csv", "wb") as f_csv:
    f_csv.write(csv_bytes)
print("Metadata Dataset_metadata.csv berhasil disimpan!")

real_infos = sorted([i for i in zf.infolist() if i.filename.startswith("Real/") and i.filename.endswith(".wav")], key=lambda x: x.filename)
fake_infos = sorted([i for i in zf.infolist() if i.filename.startswith("Fake/") and i.filename.endswith(".wav")], key=lambda x: x.filename)

def process_class(nama_kelas, infos, target_folder):
    existing = sorted([f for f in os.listdir(target_folder) if f.endswith(".wav")])
    count = len(existing)
    if count >= TARGET_JUMLAH_FILE:
        print(f"[{nama_kelas}] Sudah lengkap ({count}/{TARGET_JUMLAH_FILE} file). Lewati.")
        return

    print(f"\nMemproses kelas [{nama_kelas}] -> Target: {TARGET_JUMLAH_FILE} file @ 30 detik...")
    buffer_audio = np.array([], dtype=np.float32)

    for info in infos:
        if count >= TARGET_JUMLAH_FILE:
            break

        print(f"  Mengunduh {info.filename} ({info.compress_size/1024/1024:.1f} MB)...", end=" ", flush=True)
        wav_bytes = extract_single_member_bytes(raw_http, info)
        y, sr = librosa.load(io.BytesIO(wav_bytes), sr=SR_TARGET, mono=True)
        durasi_file = len(y) / SR_TARGET
        print(f"Selesai! (Durasi: {durasi_file:.1f} detik / {durasi_file/60:.1f} menit)")

        buffer_audio = np.concatenate([buffer_audio, y])

        # Potong per 30 detik
        while len(buffer_audio) >= SAMPEL_PER_FILE and count < TARGET_JUMLAH_FILE:
            count += 1
            potongan_30s = buffer_audio[:SAMPEL_PER_FILE]
            buffer_audio = buffer_audio[SAMPEL_PER_FILE:]
            out_path = os.path.join(target_folder, f"audio_{count:03d}.wav")
            sf.write(out_path, potongan_30s, SR_TARGET)

        print(f"  -> Progress [{nama_kelas}]: {count}/{TARGET_JUMLAH_FILE} file 30 detik tersimpan (sisa buffer: {len(buffer_audio)/SR_TARGET:.1f}s)")

# Jalankan untuk kedua kelas
process_class("ai_manipulasi", fake_infos, FOLDER_AI)
process_class("asli_human", real_infos, FOLDER_ASLI)

print("\n==========================================")
print("SELESAI! Seluruh dataset berhasil diunduh dan dipotong menjadi 30 detik.")
print("==========================================")

Menghubungkan ke server Kaggle...
Metadata Dataset_metadata.csv berhasil disimpan!
[ai_manipulasi] Sudah lengkap (100/100 file). Lewati.
[asli_human] Sudah lengkap (100/100 file). Lewati.

SELESAI! Seluruh dataset berhasil diunduh dan dipotong menjadi 30 detik.


In [8]:
# ==========================================
# VERIFIKASI HASIL POTONGAN 30 DETIK
# ==========================================
for nama_kelas, folder in [("asli_human", FOLDER_ASLI), ("ai_manipulasi", FOLDER_AI)]:
    files = sorted([f for f in os.listdir(folder) if f.endswith(".wav")])
    if files:
        p1 = os.path.join(folder, files[0])
        pn = os.path.join(folder, files[-1])
        d1 = librosa.get_duration(path=p1)
        dn = librosa.get_duration(path=pn)
        sr_cek = librosa.get_samplerate(p1)
        print(f"Kelas {nama_kelas:15s}: {len(files)} file | SR: {sr_cek} Hz | {files[0]}: {d1:.2f}s | {files[-1]}: {dn:.2f}s | Total: {len(files)*30/60:.1f} menit")

Kelas asli_human     : 100 file | SR: 16000 Hz | audio_001.wav: 30.00s | audio_100.wav: 30.00s | Total: 50.0 menit
Kelas ai_manipulasi  : 100 file | SR: 16000 Hz | audio_001.wav: 30.00s | audio_100.wav: 30.00s | Total: 50.0 menit
